# 12 · SkewerJAX and fake_spectra: the results compared

fake_spectra is the external reference code for spectra from simulations. This notebook compares what the two codes
compute on the same simulation skewers: the number density along the skewer and the transmitted flux $F = e^{-\tau}$.

fake_spectra runs natively here, called from the installed package (kernel `skewerjax-fs`, recipe in `docs/installation.md`),
and every difference between the two codes is one option of the SkewerJAX function it belongs to.

The path: one skewer through the core calls with the defaults; the same skewer in fake_spectra; the same calls with every
option at fake_spectra's value, the matched mode; then each option on its own, from both ends, and what it weighs.

In [ ]:
import contextlib, io, os, tempfile
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("OMP_NUM_THREADS", str(os.cpu_count()))  # fake_spectra's OpenMP threads
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import SWITCH_COLORS, plot_code_comparison, plot_dag, use_backend, use_style
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('simulation', 'tau', 'fake_spectra',), external=True); plt.show()

## One skewer: the SkewerJAX default

Along $y$ through the centre of the box, $x = z = L/2$, from end to end. A `SkewerBundle` holds the path and its frame,
with the snapshot's $a$ at the middle of the path and $a(x)$ from the cosmology; `select_cells_near_paths` the raw cells
within reach, `gas_to_cells` their gas physics, `cell_pieces` the pieces and `compute_optical_depth` $\tau$. Every default
is written in the calls, since they are the options of the next sections.
The pixels are fake_spectra's: $n_{\rm bins} = \lfloor v_{\max}/1\ \mathrm{km/s}\rfloor$, $dv = v_{\max}/n_{\rm bins}$,
$v_{\max} = (aH)_{\rm snap}L$

In [ ]:
from skewerjax.core import (Cells, LineBundle, SkewerBundle, SnapshotFrame, axis_aligned_skewer_points, cell_pieces, compute_optical_depth,
                            density_per_pixel, path_geometry, scale_factors_along_paths)
from skewerjax.simulations import ArepoHDF5Reader, gas_to_cells, select_cells_near_paths
from skewerjax.simulations.gas_physics import fake_spectra_sf_threshold
from skewerjax.utils.paths import DATA_DIR as DATA      # folder with the snapshots: $SKEWERJAX_DATA (docs/installation.md)

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
SPECIES = list(lines.species_keys)
LINE_OF = {"HI": 0, "SiIII": 1}

In [ ]:
SIM, SNAP, AXIS, SPACING_KMS = "TNG50-2", 25, 2, 0.25     # along y (axis 2); points every 0.25 km/s
reader = ArepoHDF5Reader(f"{DATA}/{SIM}", SNAP)
H = reader.header
L, AH, a_snap = H.box_cMpc, H.velfac_kms_per_cMpc, H.a     # [cMpc], (aH)_snap [km/s per cMpc]
nbins, dv = H.pixel_grid(1.0)                              # fake_spectra's pixels
v = (np.arange(nbins) + 0.5) * dv                          # [km/s], over [0, v_max)
V_PERIOD = nbins * dv

SNAP_FRAME = SnapshotFrame(a_snap, AH, path_length=np.array([L]))            # the frame of the matched mode

print(f"z = {H.redshift:.3f}, L = {L:.3f} cMpc, {H.n_gas:,} gas cells; {nbins} pixels of {dv:.6f} km/s")

In [ ]:
def skewers_along_y(x, z):
    # (N_sk,) x and z [cMpc]: the points from y = 0 to L and the bundle: a_snap at the middle of the path, a(x) from the cosmology
    points = axis_aligned_skewer_points(np.column_stack([x, np.zeros_like(x), z]), np.full(np.size(x), AXIS), L, SPACING_KMS / AH)
    s = np.asarray(path_geometry(jnp.asarray(points))["s"])
    a_first = scale_factors_along_paths(s, a_snap, H.cosmology(), s_ref=np.full(np.size(x), L / 2))[:, 0]
    bundle = SkewerBundle(
        spatial_points=jnp.asarray(points), a0=a_first, x0=0.0, ICs_defined_at_index=0,
        ICs_at_emission_or_absorption="absorption", cosmologies=H.cosmology()
    )
    return points, bundle

points, bundle = skewers_along_y(np.array([L / 2]), np.array([L / 2]))

print(f"{points.shape[1]:,} points; a from {float(bundle.scale_factors[0, 0, 0]):.5f} to {float(bundle.scale_factors[0, 0, -1]):.5f} (a_snap = {a_snap:.5f})")

In [ ]:
gas = select_cells_near_paths(reader, points)                                # the raw cells within reach of the path

In [ ]:
cells = gas_to_cells(
    gas, SPECIES,
    mass_unit="amu",             
    T_constants="codata",     
    x_HI_model="stored_sf",           
    cloudy_density="hydrogen",  
    cloudy_interpolation="linear",   
)
pieces, parts = cell_pieces(
    cells, lines,
    points=points,
    skewer_bundle=bundle,
    deposition="voronoi",       
    a_data=a_snap,
    return_parts=True,
)
tau = compute_optical_depth(
    pieces, v, lines,
    velocity_period=False,         
    voigt="ExoJAX",                   
    detuning="exact",                
    a_eff=None,                     
    hubble_flow="exact",
    aH_ref=None,  
    b_constants="codata",       
    pixel_average=False,               
    tautail=None,                     
    resolution_tolerance=0.25,
    piece_chunk=2048,
)

In [ ]:
T = jnp.exp(-tau)
tau0 = np.asarray(tau)

print(f"<F> = {np.exp(-tau0[0]).mean():.4f} (Lya), {np.exp(-tau0[1]).mean():.6f} (Si III)")
print(f"pixels: {nbins} of {dv:.9f} km/s, v_max {V_PERIOD:.6f}")

In [ ]:
def densities(parts):
    # {species: (N_sk, nbins)} n [physical cm^-3 at the snapshot] on fake_spectra's pixels (below)
    return {sp: density_per_pixel(parts["absorbers"][sp], L, nbins) for sp in SPECIES}

y = (np.arange(nbins) + 0.5) * L / nbins                       # [cMpc], the centres of the position pixels of n
rms = lambda d: float(np.sqrt(np.nanmean(np.square(d))))

def result(tau, n):
    # {species: n, tau, F on the central skewer, with their grids}: one mode, as the figures take it
    tau = np.asarray(tau)
    return {sp: dict(n=n[sp][0], tau=tau[LINE_OF[sp], 0], F=np.exp(-tau[LINE_OF[sp], 0]), v=v, y=y) for sp in SPECIES}

n0 = densities(parts)
DEFAULT = result(tau0, n0)

The number density is the mean physical density in equal **position** pixels of the path, at the snapshot
(`density_per_pixel`: the exact overlaps of the pieces with each pixel, over its physical length), so its axis is the
comoving position $y$. The figures draw it above $\tau$ and $F$, which are against $v$, with the same horizontal extent
through $y = v/(aH)_{\rm snap}$.

## The same skewer in fake_spectra

`Spectra` is called directly, on the same snapshot and sightline and with the same pixel width. Its sightline is an axis
(1, 2, 3 = $x$, $y$, $z$) and a point in code units (ckpc/$h$): printed next to SkewerJAX's in cMpc so they can be
compared. Its options stay at its defaults for TNG: `sf_neutral=True` and the top-hat kernel.
It also offers a partial reconstruction of the Voronoi mesh, `kernel="voronoi"`.
$n$ is `get_density`, `get_col_density` over the physical pixel length, the same quantity as above.

In [ ]:
from fake_spectra.spectra import Spectra
WORK = tempfile.mkdtemp(prefix="fs_work_")

def quiet(f, *args):
    # fake_spectra prints its progress per file of the snapshot, whatever quiet=
    with contextlib.redirect_stdout(io.StringIO()):
        return f(*args)

def native_spectra(x, z, name):
    # Spectra for the sightlines along y at (x, z) [cMpc]: axis and cofm in its units
    cofm = np.column_stack([x, np.zeros_like(x), z]) / H.code_length_cMpc
    return Spectra(SNAP, f"{DATA}/{SIM}", cofm, np.full(np.size(x), AXIS, dtype=np.int32), res=1.0, savefile=f"{name}.hdf5", savedir=WORK, reload_file=True, quiet=True)

In [ ]:
fs = native_spectra(np.array([L / 2]), np.array([L / 2]), "centre")
u = (np.arange(fs.nbins) + 0.5) * fs.dvbin                     # [km/s], fake_spectra's own velocity pixels
FS = {
    "HI": dict(tau=np.asarray(quiet(fs.get_tau, "H", 1, 1215))[0], n=np.asarray(quiet(fs.get_density, "H", 1))[0]),
    "SiIII": dict(tau=np.asarray(quiet(fs.get_tau, "Si", 3, 1206))[0], n=np.asarray(quiet(fs.get_density, "Si", 3))[0])
}
for f in FS.values():
    f.update(F=np.exp(-f["tau"]), v=u, y=y)
    
print(f"{os.environ['OMP_NUM_THREADS']} OpenMP threads; kernel {fs.kernel_int} (0 = top hat), sf_neutral = {fs.gasprop.sf_neutral}")
print(f"sightline, fake_spectra: axis {fs.axis[0]}, cofm {fs.cofm[0]} ckpc/h = {fs.cofm[0] * H.code_length_cMpc} cMpc")
print(f"pixels: {fs.nbins} of {fs.dvbin:.9f} km/s, v_max {fs.vmax:.6f}; max |u - v| = {np.abs(u - v).max():.1e} km/s")

## One skewer: the SkewerJAX matched

The same calls as in the default block, with every option written at fake_spectra's value and a comment on what it changes.
The next section takes these options one by one.

In [ ]:
cells_m = gas_to_cells(
    gas, SPECIES,
    mass_unit="proton",             # n = rho X / (A m_p), not A m_u
    T_constants="fake_spectra",     # its rounded m_p and k_B in T
    x_HI_model="fake_sf",           # sf_neutral=True: its density threshold and its UVB
    cloudy_density="fake_spectra",  # hden = 0.774 rho / m_p, not n_H
    cloudy_interpolation="cubic",   # a spline of log f, not bilinear
)
pieces_m, parts_m = cell_pieces(
    cells_m, lines,
    points=points,
    frame=SNAP_FRAME,                 # one a and u = aH s + v_pec, not the bundle's a(x)
    deposition="tophat_chords",       # spheres of radius V^(1/3) and 7 points per chord, not the Voronoi cells
    chord_weight=1 / 8,               # its SPH weights: 7/8 of the column, in tau only
    a_data=a_snap, return_parts=True,
)
tau_m = compute_optical_depth(
    pieces_m, v, lines,
    velocity_period=V_PERIOD,         # periodic by construction, not a direct integral
    voigt="harris",                   # the first order in the damping parameter, not ExoJAX
    detuning="linear",                # H(a, du / b), not the exact map to frequency
    a_eff=a_snap,                     # one comoving-to-physical conversion, not a_p^-2 along the path
    hubble_flow="linear",
    aH_ref=AH,                        # the line centres on (aH)_snap y, not c ln(a_dagger / a(y))
    b_constants="fake_spectra",       # b^2 = 2 k_B T / (A m_p) with the constants of its C engine
    pixel_average=True,               # cells with b < 2 dv averaged over the pixel, not the profile at its centre
    tautail=1e-7,                     # each cell stops below 1e-7, not its full profile
    resolution_tolerance=None, piece_chunk=2048,
)

print(f"<F> = {np.exp(-np.asarray(tau_m[0])).mean():.4f} (Lya), {np.exp(-np.asarray(tau_m[1])).mean():.6f} (Si III)")

In [ ]:
def tophat_density(cells):
    # n when tau uses the chord points: fake_spectra's column keeps the exact overlaps of the spheres (absorption.cpp L211)
    return densities(cell_pieces(cells, lines, skewer_bundle=bundle, deposition="tophat", a_data=a_snap, return_parts=True)[1])

MATCHED = result(tau_m, tophat_density(cells_m))

In [ ]:
ZOOM = (V_PERIOD-1000, V_PERIOD)      # the window of every figure below: None (the whole skewer) or e.g. (1500, 2500) [km/s]
for sp in SPECIES:
    fig, axs = plot_code_comparison(
        [dict(mode="fake_spectra", **FS[sp]), dict(mode="default", **DEFAULT[sp]), dict(mode="matched", **MATCHED[sp])],
        residuals=[("default", "fake_spectra"), ("matched", "fake_spectra")], linewidths=(5, 4, 3), residual_limits=dict(dF=(1e-7, 1e-15)) if sp == "SiIII" else None, aH=AH, species=sp, window=ZOOM,
        title=f"{lines.lines[LINE_OF[sp]].id}: the matched mode reproduces fake_spectra, rms(ΔF) = {rms(MATCHED[sp]['F'] - FS[sp]['F']):.1e}, against {rms(DEFAULT[sp]['F'] - FS[sp]['F']):.1e} for the default")
    plt.show()

## The switches, one figure each

Every difference is one option, and each is tested on its own, from both ends:

- **A** is the default with only this option at fake_spectra's value: what this choice does to SkewerJAX;
- **B** is the matched mode with only this option back at SkewerJAX's default: how far fake_spectra's result moves without it.

Nothing accumulates: every step starts again from the default and from the matched mode above. The steps follow the route of the gas:

1. **Extraction** (`gas_to_cells`, steps 1–5): the gas physics per cell, before any geometry. The constants first, then
   the ionisation that depends on them.
2. **Deposition** (`cell_pieces`, steps 6–8): where that gas sits along the path.
3. **Integration** (`compute_optical_depth`, steps 9–17): first what the profile of one piece is ($b$, Voigt), then the
   spectral axis (period, frame, detuning), and last the two per-cell schemes on that axis: the pixel average, and
   `tautail`, whose walk runs over the pixel averages.

A few options need another one, and take it along in both directions, marked `*` in the cell and the title:

| step | A takes along | B takes along |
|---|---|---|
| 6, `deposition="tophat"` | | `chord_weight`: without spheres there are no chords |
| 7, `deposition="tophat_chords"` | the spheres: the chords are chords of them | `chord_weight`: without chord points there are no weights |
| 8, `chord_weight=1/8` | `deposition="tophat_chords"` | |
| 12, `a_eff` | | the `ExactFrame`: in `SnapshotFrame`, $a = a_{\rm snap}$ everywhere, so `a_eff=None` changes nothing |
| 13, `hubble_flow` | | the `ExactFrame`: in `SnapshotFrame` the flow is in the velocities, so `"exact"` changes nothing |
| 14, `SnapshotFrame` | steps 12 and 13 | |

Each cell has the same calls as above, every option written: first A, then B, with the option marked. Each figure has four
curves, the default, the matched mode, A and B, and two residuals, A against the default and B against the matched mode. fake_spectra is not drawn: the
matched mode stands for it, since it reproduces it to $\sim10^{-6}$ in $F$ (above). For $n$ and $\tau$ the residual is $2(a-b)/(a+b)$ over the pixels with gas, written
$\Delta n/\bar n$ and $\Delta\tau/\bar\tau$; for $F$ it is $\Delta F$. The residual rows have the same limits in all the figures.

Each curve of $\tau$ and $F$ is on the $v$ of its own mode, which is $c\ln(a_\dagger/a(y))$ until `hubble_flow="linear"` (step 13). The pixels are the same numbers, the features differ
slightly, and this is left uncorrected: it is one of the differences being measured.

In [ ]:
A, B = {}, {}    # {switch: result(...)}: A the default + this switch, B the matched mode - this switch

def step_figure(step, switch, species="HI", star=False):
    # the four curves and two residuals of one step; the matched mode stands for fake_spectra
    a, b, j = A[switch][species], B[switch][species], LINE_OF[species]
    fig, axs = plot_code_comparison(
        [dict(mode="default", **DEFAULT[species]), dict(mode="matched", **MATCHED[species]),
         dict(mode="default+switch", **a), dict(mode="matched-switch", **b)],
        residuals=[("default+switch", "default"), ("matched-switch", "matched")], linewidths=(4, 3.5, 3, 2.5),
        aH=AH, species=species, window=ZOOM,
        title=f"Step {step}, {switch}{' *' if star else ''}, {lines.lines[j].id}: rms(ΔF) = {rms(a['F'] - DEFAULT[species]['F']):.2e} (A − default), {rms(b['F'] - MATCHED[species]['F']):.2e} (B − matched)")
    plt.show()

**Step 1, `T_constants`.** fake_spectra's rounded $m_p$ and $k_B$ in $T = (\gamma-1)\mu m_p u/k_B$ (`unitsystem.py` L21, L23; `abstractsnapshot.py` L121): $T$ lower by $8\times10^{-6}$.

In [ ]:
# step 1: T_constants
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="fake_spectra", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)
A["T_constants"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="codata", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)
B["T_constants"] = result(tau, tophat_density(cells))
step_figure(1, "T_constants")

**Step 2, `mass_unit`.** $n = \rho X/(A m_p)$ instead of $A m_u$: every density down by 0.73 % (`gas_properties.py` L108; `absorption.h` L4 calls the proton mass "1 a.m.u").

In [ ]:
# step 2: mass_unit
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)
A["mass_unit"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)
B["mass_unit"] = result(tau, tophat_density(cells))
step_figure(2, "mass_unit")

**Step 3, `x_HI_model`.** fake_spectra's default `sf_neutral=True` (`spectra.py` L86, `gas_properties.py` L116–L146): above its own threshold in $\rho/m_p$, Rahmati+13 at $10^4$ K with its FG09 rate; SkewerJAX recomputes the cells with SFR > 0, with the simulation's UVB. Only H I; the cell prints how many cells this can touch on this skewer.

In [ ]:
# step 3: x_HI_model
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="fake_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)
A["x_HI_model"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="stored_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)
B["x_HI_model"] = result(tau, tophat_density(cells))
step_figure(3, "x_HI_model")
rho_mp = np.asarray(gas.fields["rho"]) / 1.67262178e-24
print(f"of the {len(gas.positions):,} cells near this skewer, {int((np.asarray(gas.fields['sfr']) > 0).sum())} have SFR > 0, {int((rho_mp > fake_spectra_sf_threshold(H.h) / 0.76).sum())} are above fake_spectra's threshold; densest rho / m_p = {rho_mp.max():.3g} cm^-3")

**Step 4, `cloudy_density`.** The density coordinate of the CLOUDY table, $h_{\rm den} = 0.774\,\rho/m_p$ instead of $n_{\rm H}$ (`spectra.py` L691–L698, `convert_cloudy.py` L183). Only metals: this figure shows Si III. On this skewer Si III is optically thin by many decades, so its effect is in $n$ and $\tau$, not in $F$.

In [ ]:
# step 4: cloudy_density
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="fake_spectra", cloudy_interpolation="linear")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)
A["cloudy_density"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="hydrogen", cloudy_interpolation="cubic")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)
B["cloudy_density"] = result(tau, tophat_density(cells))
step_figure(4, "cloudy_density", species="SiIII")

**Step 5, `cloudy_interpolation`.** A cubic spline of $\log f$ instead of bilinear (`convert_cloudy.py` L199). Only metals: Si III again.

In [ ]:
# step 5: cloudy_interpolation
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="cubic")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)
A["cloudy_interpolation"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="linear")  # <-- this switch
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)
B["cloudy_interpolation"] = result(tau, tophat_density(cells))
step_figure(5, "cloudy_interpolation", species="SiIII")

**Step 6, `deposition="tophat"`.** Spheres of radius $V^{1/3}$ and density $3n/4\pi$ (`absorption.cpp` L112–L118, `abstractsnapshot.py` L256–L304), their exact overlaps with the path, instead of the Voronoi cells. The one switch that moves gas between pixels.

In [ ]:
# step 6: deposition='tophat' *
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="tophat", chord_weight=None, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)
A["deposition='tophat'"] = result(tau, densities(parts))
# B: the matched mode - this switch; * chord_weight goes with the chords: the Voronoi pieces have neither
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=0.25, piece_chunk=2048)
B["deposition='tophat'"] = result(tau, densities(parts))
step_figure(6, "deposition='tophat'", star=True)

**Step 7, `deposition="tophat_chords"`.** The same spheres through fake_spectra's quadrature for the optical depth: 7 interior points per chord (`singleabs.h` L100–L118), here each of weight 1/7, so the column stays. $n$ is that of step 6: fake_spectra's column keeps the exact overlaps (`absorption.cpp` L211).

In [ ]:
# step 7: deposition='tophat_chords' *
# A: the default + this switch; * the spheres come with the chords
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="tophat_chords", chord_weight=None, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=None, piece_chunk=2048)
A["deposition='tophat_chords'"] = result(tau, tophat_density(cells))
# B: the matched mode - this switch; * chord_weight goes with the chords: exact overlaps have no weights
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat", chord_weight=None, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=0.25, piece_chunk=2048)
B["deposition='tophat_chords'"] = result(tau, densities(parts))
step_figure(7, "deposition='tophat_chords'", star=True)

**Step 8, `chord_weight=1/8`.** Its weights: $\Delta v = 2v_{\rm high}/8$ (`singleabs.h` L100, L116), meant for an SPH kernel that vanishes at both ends (L97–L99). A top hat then integrates 7/8 of its column, in $\tau$ only (not in `colden`, `absorption.cpp` L211).

In [ ]:
# step 8: chord_weight=1/8 *
# A: the default + this switch; * needs the chords of deposition='tophat_chords' (step 7)
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=None, piece_chunk=2048)
A["chord_weight=1/8"] = result(tau, tophat_density(cells))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=None, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)
B["chord_weight=1/8"] = result(tau, tophat_density(cells))
step_figure(8, "chord_weight=1/8", star=True)

**Step 9, `b_constants`.** $b^2 = 2k_BT/(A m_p)$ with the constants of its C engine (`absorption.cpp` L24, L158), 0.72 % below $2k_BT/(A m_u)$.

In [ ]:
# step 9: b_constants
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="fake_spectra", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["b_constants"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="codata", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)  # <-- this switch
B["b_constants"] = result(tau, tophat_density(cells))
step_figure(9, "b_constants")

**Step 10, `voigt="harris"`.** The first order in the damping parameter, $H = e^{-u^2} + a H_1(u)$, exact above $a = 10^{-3}$ (`singleabs.h` L57–L66, `fastvoigt.h` L62–L80).

In [ ]:
# step 10: voigt
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="harris", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["voigt"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="ExoJAX", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)  # <-- this switch
B["voigt"] = result(tau, tophat_density(cells))
step_figure(10, "voigt")

**Step 11, `velocity_period`.** fake_spectra's spectrum is periodic by construction, its indices wrap (`absorption.cpp` L261–L264): the wings that leave one end re-enter at the other.

In [ ]:
# step 11: velocity_period
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["velocity_period"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)  # <-- this switch
B["velocity_period"] = result(tau, tophat_density(cells))
step_figure(11, "velocity_period")

**Step 12, `a_eff`.** One comoving-to-physical conversion at $a_{\rm snap}$ instead of $a_p^{-2}$ along the path (`spectra.py` L221, L615).

In [ ]:
# step 12: a_eff *
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=a_snap, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["a_eff"] = result(tau, densities(parts))
# B: the matched mode - this switch; * SnapshotFrame has a = a_snap everywhere, so a_eff=None alone changes nothing: the ExactFrame comes back too
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=None, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)  # <-- this switch
B["a_eff"] = result(tau, tophat_density(cells))
step_figure(12, "a_eff", star=True)

**Step 13, `hubble_flow="linear"`.** The line centres on the linear flow $(aH)_{\rm snap}\,y$ instead of $c\ln(a_\dagger/a(y))$ (`absorption.cpp` L237; `spectra.py` L225–L228). The largest shift of the features.

In [ ]:
# step 13: hubble_flow *
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="linear", aH_ref=AH, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["hubble_flow"] = result(tau, densities(parts))
# B: the matched mode - this switch; * SnapshotFrame has its flow in the velocities, so 'exact' alone changes nothing: the ExactFrame comes back too
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="exact", aH_ref=None, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)  # <-- this switch
B["hubble_flow"] = result(tau, tophat_density(cells))
step_figure(13, "hubble_flow", star=True)

**Step 14, `SnapshotFrame`.** What is left of the frame beyond steps 12 and 13: the peculiar velocity added to $u$ as it is, $u = aHs + v_{\rm pec}$ (`absorption.cpp` L237), instead of the relativistic $c\,\mathrm{artanh}(v/c)$.

In [ ]:
# step 14: frame='snapshot' *
# A: the default + this switch; * needs a_eff and hubble_flow (steps 12, 13)
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["frame='snapshot'"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)  # <-- this switch
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)
B["frame='snapshot'"] = result(tau, tophat_density(cells))
step_figure(14, "frame='snapshot'", star=True)

**Step 15, `detuning="linear"`.** $H(a, \Delta u/b)$ (`singleabs.h` L182), the first order of $\nu = (c/\lambda)\,\mathrm{expm1}(-\Delta v/c)$.

In [ ]:
# step 15: detuning
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="linear", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["detuning"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="exact", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)  # <-- this switch
B["detuning"] = result(tau, tophat_density(cells))
step_figure(15, "detuning")

**Step 16, `pixel_average`.** A cell narrower than twice the pixel, $b < 2\,dv$, is averaged over the pixel on a trapezoid (`singleabs.h` L133–L155).

In [ ]:
# step 16: pixel_average
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=True, tautail=None, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["pixel_average"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=False, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048)  # <-- this switch
B["pixel_average"] = result(tau, tophat_density(cells))
step_figure(16, "pixel_average")

**Step 17, `tautail`.** Walking out from each cell's line centre, stop after the first pixel that receives less than $10^{-7}$ from that cell (`absorption.cpp` L253–L281, `spectra.py` L145).

In [ ]:
# step 17: tautail
# A: the default + this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="amu", T_constants="codata", x_HI_model="stored_sf", cloudy_density="hydrogen", cloudy_interpolation="linear")
pieces, parts = cell_pieces(cells, lines, skewer_bundle=bundle, deposition="voronoi", chord_weight=None, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=False, voigt="ExoJAX", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None, b_constants="codata", pixel_average=False, tautail=1e-7, resolution_tolerance=0.25, piece_chunk=2048)  # <-- this switch
A["tautail"] = result(tau, densities(parts))
# B: the matched mode - this switch
cells = gas_to_cells(gas, SPECIES, mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra", cloudy_interpolation="cubic")
pieces, parts = cell_pieces(cells, lines, points=points, frame=SNAP_FRAME, deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap, return_parts=True)
tau = compute_optical_depth(pieces, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH, b_constants="fake_spectra", pixel_average=True, tautail=None, resolution_tolerance=None, piece_chunk=2048)  # <-- this switch
B["tautail"] = result(tau, tophat_density(cells))
step_figure(17, "tautail")

In [ ]:
STARRED = {"deposition='tophat'", "deposition='tophat_chords'", "chord_weight=1/8", "a_eff", "hubble_flow", "frame='snapshot'"}
mark = lambda switch: switch + (" *" if switch in STARRED else "")
MASTER_WINDOW = {"HI": ZOOM, "SiIII": (1500, 1800)}     # [km/s]; None: the whole skewer
LINE_LIMITS = {"HI": None, "SiIII": dict(dF=(1e-7, 1e-15))}  # (limit, linear threshold) of the residual rows, per line
THIN = dict(ls="-", lw=1.0)
print(f"Si III in fake_spectra: max tau = {FS['SiIII']['tau'].max():.1e}, {np.mean(FS['SiIII']['tau'] == 0):.0%} of the pixels exactly 0; "
      f"matched {np.mean(MATCHED['SiIII']['tau'] == 0):.0%}, matched without tautail {np.mean(B['tautail']['SiIII']['tau'] == 0):.0%}")

for sp in SPECIES:
    name, window = lines.lines[LINE_OF[sp]].id, MASTER_WINDOW[sp]
    where = "" if window is None else f", {window[0]}–{window[1]} km/s"
    by_size = sorted(A, key=lambda s: rms(A[s][sp]["F"] - DEFAULT[sp]["F"]))
    fig, axs = plot_code_comparison(
        [dict(mode="default", **DEFAULT[sp]), *(dict(mode="default+switch", label=mark(s), color=SWITCH_COLORS[s], lw=1.2, **A[s][sp]) for s in A)],
        residuals=[(mark(s), "default", THIN) for s in by_size], residual_note="each switch − default", residual_limits=LINE_LIMITS[sp],
        aH=AH, species=sp, window=window, legend_ncols=3, title=f"{name}{where}: each switch alone on the SkewerJAX default")
    plt.show()
    by_size = sorted(B, key=lambda s: rms(B[s][sp]["F"] - FS[sp]["F"]))
    fig, axs = plot_code_comparison(
        [dict(mode="matched", **MATCHED[sp]), dict(mode="fake_spectra", **FS[sp]),
         *(dict(mode="matched-switch", label=mark(s), color=SWITCH_COLORS[s], lw=1.2, **B[s][sp]) for s in B)],
        residuals=[*((mark(s), "fake_spectra", THIN) for s in by_size), ("matched", "fake_spectra", THIN)], residual_note="each − fake_spectra", residual_limits=LINE_LIMITS[sp],
        aH=AH, species=sp, window=window, legend_ncols=3, title=f"{name}{where}: the matched mode without each switch, against fake_spectra")
    plt.show()

### The fully matched mode: what remains

Every option at fake_spectra's value, against fake_spectra itself, on the whole skewer and on 1000 km/s of it, for both
lines. Both codes integrate in double precision, so what remains comes from the single precision of fake_spectra's gas
fields. Its largest part is systematic: $h = V^{1/3}$ is `np.power(V, 1./3)` on a float32 array (`abstractsnapshot.py`
L271), with the exponent rounded up in float32, so $h$ is $1.5\times10^{-7}$ high and so are the chords and $\tau$. Giving
SkewerJAX that same $h$ removes it (printed below). One case stands apart, not on these TNG50-2 skewers but on a TNG50-4
skewer through a damped absorber ($\tau_{\max} \sim 5\times10^{6}$): the single pixel half a box from it differs by
$|\Delta F| = 1.6\times10^{-5}$, because SkewerJAX takes the nearest image per chord point and fake_spectra per cell.

In [ ]:
for sp in SPECIES:
    F_m, d = MATCHED[sp]["F"], None
    d = F_m - FS[sp]["F"]
    has = FS[sp]["n"] > 0
    dn = MATCHED[sp]["n"][has] / FS[sp]["n"][has] - 1
    print(f"{lines.lines[LINE_OF[sp]].id:11s} max |dF| = {np.abs(d).max():.2e}, rms dF = {np.sqrt(np.mean(d**2)):.2e}, <F> {F_m.mean():.6f} against {FS[sp]['F'].mean():.6f}; "
          f"max tau {FS[sp]['tau'].max():.3g}; n: rms relative {np.sqrt(np.mean(dn**2)):.2e}, max {np.abs(dn).max():.2e}")
V_code = np.asarray(cells_m.volumes) / H.code_length_cMpc**3                                  # the cells of the matched mode
h32 = np.power(V_code.astype(np.float32), 1.0 / 3.0).astype(np.float64) * H.code_length_cMpc  # fake_spectra's h, in float32
pieces32 = cell_pieces(Cells(np.asarray(cells_m.positions), h32**3, cells_m.species, box=cells_m.box, ids=cells_m.ids), lines, points=points, frame=SNAP_FRAME,
                       deposition="tophat_chords", chord_weight=1 / 8, a_data=a_snap)
tau32 = np.asarray(compute_optical_depth(pieces32, v, lines, velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH,
                                         b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None, piece_chunk=2048))
m = (FS["HI"]["tau"] > 1e-3) & (FS["HI"]["tau"] < 5)
for name, t in (("as it is", MATCHED["HI"]["tau"]), ("with fake_spectra's float32 h", tau32[0, 0])):
    print(f"Lya, {name}: median relative d tau = {np.median((t - FS['HI']['tau'])[m] / FS['HI']['tau'][m]):+.2e}, median |dF| = {np.median(np.abs(np.exp(-t) - FS['HI']['F'])):.2e}")
pair = [("matched", "fake_spectra")]
for sp, window, about in (("HI", (1500, 2500), "on 1000 km/s of the skewer"), ("SiIII", ZOOM, "Si III 1206")):
    fig, axs = plot_code_comparison([dict(mode="fake_spectra", **FS[sp]), dict(mode="matched", **MATCHED[sp])], residuals=pair, linewidths=(4, 1), residual_limits=LINE_LIMITS[sp], aH=AH, species=sp, window=window,
                                    title=f"The fully matched mode, {about}: rms(ΔF) = {rms(MATCHED[sp]['F'] - FS[sp]['F']):.1e} over the whole skewer"); plt.show()

## Root causes, and why SkewerJAX's choice is the more accurate one

Each row is an option of a SkewerJAX function; the files and lines of fake_spectra 2.3.0 (`.project_material/fake_spectra`) are in the steps above.
The numbers are what A, the default with that one switch, changes on this skewer: $\mathrm{rms}(\Delta F)$ of each line.
The rank orders the switches by the larger $\mathrm{rms}(\Delta F)$ of the two lines.

In [ ]:
from IPython.display import Markdown, display

ROOT = [  # (switch, the option at fake_spectra's value, what fake_spectra does, why SkewerJAX's choice is more accurate), by step
    ('T_constants', '`gas_to_cells(T_constants="fake_spectra")`', 'rounded $m_p$, $k_B$ in $T$', 'CODATA 2018'),
    ('mass_unit', '`gas_to_cells(mass_unit="proton")`', '$n = \\rho X/(A m_p)$', '$A$ is an atomic weight in units of $m_u$; $m_p/m_u = 1.00728$'),
    ('x_HI_model', '`gas_to_cells(x_HI_model="fake_sf")`', 'a density threshold in $\\rho/m_p$, its own UVB table', "TNG's own SFR > 0 flag and its UVB (TREECOOL): the cold phase only where the simulation formed stars"),
    ('cloudy_density', '`gas_to_cells(cloudy_density="fake_spectra")`', '$h_{\\rm den} = 0.774\\,\\rho/m_p$', "$h_{\\rm den} = n_{\\rm H}$, CLOUDY's own coordinate"),
    ('cloudy_interpolation', '`gas_to_cells(cloudy_interpolation="cubic")`', 'a cubic spline of $\\log f$', 'linear in $\\log f$ is monotonic; the spline overshoots near the $-30$ floor of the table'),
    ("deposition='tophat'", '`cell_pieces(deposition="tophat")`', 'each cell as a sphere of radius $V^{1/3}$, density $3n/4\\pi$', 'Arepo solves the hydrodynamics on the Voronoi cells: the nearest generator is the simulated fluid, piecewise constant and single valued; spheres overlap and leave gaps'),
    ("deposition='tophat_chords'", '`cell_pieces(deposition="tophat_chords")`', '7 points per chord', 'the pieces are exact intersections: no quadrature of the density'),
    ('chord_weight=1/8', '`cell_pieces(chord_weight=1/8)`', 'SPH weights on a top hat: 7/8 of the column in $\\tau$ (not in `colden`)', 'the column of a cell is a property of the gas: the exact lengths conserve it for any sampling'),
    ('b_constants', '`compute_optical_depth(b_constants="fake_spectra")`', '$b^2 = 2k_BT/(A m_p)$', 'CODATA and $m_u$'),
    ('voigt', '`compute_optical_depth(voigt="harris")`', '$H = e^{-u^2} + aH_1(u)$ for $a \\leq 10^{-3}$', 'ExoJAX to $\\sim10^{-6}$ against `wofz`; the expansion is as good where $a^2$ is small, so this is a matter of speed'),
    ('velocity_period', '`compute_optical_depth(velocity_period=...)`', 'periodic by construction', 'a path is not periodic in velocity by nature: the direct integral, opt-in period'),
    ('a_eff', '`compute_optical_depth(a_eff=a_snap)`', 'one comoving-to-physical conversion', '$a(x)$ along the path: the density of each piece at its own epoch'),
    ('hubble_flow', '`compute_optical_depth(hubble_flow="linear", aH_ref=...)`', '$u = (aH)_{\\rm snap}s$', '$c\\ln(a_\\dagger/a(y))$ from the cosmology: the expansion inside the box shifts the line centres'),
    ("frame='snapshot'", '`SnapshotFrame` in `cell_pieces`', '$v_{\\rm pec}$ added to $u$', 'the relativistic composition; the difference is $(v/c)^2 v/3$'),
    ('detuning', '`compute_optical_depth(detuning="linear")`', '$H(a, \\Delta u/b)$', 'the exact map $\\nu = (c/\\lambda)\\,\\mathrm{expm1}(-\\Delta v/c)$'),
    ('pixel_average', '`compute_optical_depth(pixel_average=True)`', "a cell's profile averaged over the pixel when $b < 2\\,dv$", 'not an accuracy: SkewerJAX gives $F$ at the pixel centres, a convention; narrow lines need finer pixels either way'),
    ('tautail', '`compute_optical_depth(tautail=1e-7)`', "stops each cell's profile below $10^{-7}$", 'full Voigt profiles: the wings of a strong absorber and of many weak cells add up'),
]

num = lambda x: "0" if x == 0 else f"{x:.1e}"
dF = {s: {sp: rms(A[s][sp]["F"] - DEFAULT[sp]["F"]) for sp in SPECIES} for s in A}
size = {s: max(dF[s].values()) for s in A}
rank = {s: 1 + sum(size[o] > size[s] for o in A) for s in A}
head = ["rank", "step", "option (fake_spectra's value)", "what fake_spectra does", "why SkewerJAX's choice is more accurate",
        "rms(ΔF) Lyα", "rms(ΔF) Si III"]
out = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
for step, (s, option, does, why) in enumerate(ROOT, 1):
    star = " \\*" if s in STARRED else ""
    out.append(f"| {rank[s]} | {step}{star} | {option} | {does} | {why} | {num(dF[s]['HI'])} | {num(dF[s]['SiIII'])} |")
own = {"deposition='tophat_chords'": ("A(7) − A(6)", A["deposition='tophat_chords'"], A["deposition='tophat'"]),
       "chord_weight=1/8": ("A(8) − A(7)", A["chord_weight=1/8"], A["deposition='tophat_chords'"]),
       "frame='snapshot'": ("B(14) − matched", B["frame='snapshot'"], MATCHED)}
notes = "; ".join(f"`{s}`, {name}: {num(rms(a['HI']['F'] - b['HI']['F']))} (Lyα), {num(rms(a['SiIII']['F'] - b['SiIII']['F']))} (Si III)" for s, (name, a, b) in own.items())
out += ["", f"\\* takes along what it needs (the table of the switches). In steps 7, 8 and 14, that is in A too, so their numbers and rank include it. Their own rms(ΔF): {notes}."]
display(Markdown("\n".join(out)))